# Software Engineering (an introduction)

James Percival

### Learning Outcomes:
By the end of today, you should be able to:
 - Explain the basics of code testing & some ways to automate it.
 - Describe a variety of sorts of test and compare between them.
 - Explain how to use Sphinx to generate software documentation from docstrings

## Software engineering

Software gets built in a number of ways:
- "garage" programming (one person working alone)
- A small team (about 6-10)
- A large team (perhaps 20-40 people)
- Entire company (100s of people)

Methods have been developed to _manage_ and _assist_ in this process

- Project management : [_waterfall_](https://www.atlassian.com/agile/project-management/waterfall-methodology), [_Agile_](https://www.atlassian.com/agile/project-management), [_iterative_](https://www.tutorialspoint.com/sdlc/sdlc_iterative_model.htm), [_scrum_](https://www.scrum.org/resources/what-scrum-module)
- Software testing & quality control
- Version control & continuous integration

## Code Testing

When writing code (especially for scientific, medical or military concerns) a key question is whether it is actually is correct, & (ideally) provably correct.

In science, how do we check something for correctness? We test it.

### What is a test?

Simplest kind of code test is the ad hoc kind you run when first hacking about with code. If I write a new function, `add(x, y)` which adds together two numbers, I might try tests

```python
print(f'1 + 0 = 0: {add(1, 0) == 1}')
print(f'1 + 1 = 2: {add(1, 1) == 2}')
```

This approach can't catch every possible failure. For example, an `add` defined function like
```python
def add(x, y):
    return x**2+y**2
```
would pass both the tests given above.

The examples above are both  "test to pass" - we have written a statement we expect to be true, and will be happy if it does so. Should check that operations you expect to raise an exception actually do  (i.e. testing to fail).

Usually ways of writing a test to fail as a test to pass if needed. E.g. for a `divide` operator:

In [ ]:
def divide(x, y):
    """return x divided by y."""
    return x / y


## Check we get an error when dividing by zero.
try:
    divide(1, 0)
    print(False)
except ZeroDivisionError:
    print(True)

By writing our tests like this we can use a statement like `assert X` instead of `print(False)` to turn the test from raising an exception if when it does what we want into raising an exception if it doesn't.

## Test Driven Development

Most extreme version of this approachis Test Driven Development (TDD), sometimes  "Red-Green" testing.

- To write new code/functional, *first* you write a test for it, which fails (red)
- *second* you write only enough code to pass the test you've just written (green).
- Finally refactor (i.e. rewrite) any code which can be improved (simplified/sped up), _while making sure tests still pass_.

This is meant to be an iterative process:

- once you are done with one test, you move on to the next.

- Let's have an example, pushing things as far as they can go:

__Problem__: Write a function that returns the repeated elements in a list.

So, first, we write a test.

```python
assert(f([0, 1, 1]) == 1)
```

Does it fail on a do nothing implementation?

In [ ]:
def f(x):
    """Return the repeated elements in a list."""
    pass


print(f([0, 1, 1]) == 1)

Test is failing, let's actually write some code. Remember that a quick way to check for repeated elements is to use a set.

In [ ]:
def f(x):
    """Return the repeated elements in a list."""
    vals = set()
    for _ in x:
        if _ in vals:
            return _
        else:
            vals.add(_)


print(f([0, 1, 1]) == 1)

Our test is now passing. There's no old code to improve, so we either write another test for additional functionality, or move on to other things. Let's try having a couple of repeated elements

In [ ]:
# Our tests:
print(f([0, 1, 1]) == 1)
print(f([0, 1, 1, 0]) == [0, 1])

In [ ]:
def f(x):
    """Return the repeated elements in a list."""
    vals = set()
    out = []
    for _ in x:
        if _ in vals:
            out.append(_)
        else:
            vals.add(_) 
    return sorted(out)


print(f([0, 1, 1]) == 1)
print(f([0, 1, 1, 0]) == [0, 1])

We've broken the first test. But it's actually the test which is bad (it's more consistent to always return a list, or similar datatype), so we'll fix the test, not write more code.

In [ ]:
# Our tests: Note first line is changed
print(f([0, 1, 1]) == [1])
print(f([0, 1, 1, 0]) == [0, 1])

Ok. Another iteration. What about if an element turns up 3 times?

In [ ]:
print(f([0, 1, 1, 0, 0]) == [0, 1])
# What's wrong with this implementation?

In [ ]:
# Start debugging by looking at the output
print(f([0, 1, 1, 0, 0]))

In [ ]:
def f(x):
    """Return the set of repeated elements in a list."""
    vals = set()
    out = set()
    for _ in x:
        if _ in vals:
            out.add(_)
        else:
            vals.add(_)
    return out

In [ ]:
# Our next set of tests
# Note the refactoring
print(f([0, 1, 1]) == set([1]))
print(f([0, 1, 1, 0]) == set([0, 1]))
print(f([0, 1, 1, 0, 0]) == set([0, 1]))

And so on. At some point you run out of tests to set/functionality to input and the code is finished.

At this point you can hopefully see that
1. TDD is a fine ideal for correct software engineering
2. No sane person would write all their tests at that level of detail shown and in that order all the time.

In real life, tests are often at a slightly higher level, and the functions more complicated.

Hints at a good philosophy to direct your thinking:
- Coding as a contract:
  - what does my function require?
  - what does my function promise?

Still some very important ideas to pull out of the paradigm:

1. Make sure your **tests can fail** if things go wrong.
2. Consider both most common case and the (obvious) corner cases you can think of.
3. Keep each test as small as you can.
4. Each bug you find and fix is a missing test you might need to add _somewhere_. Especially once you fix it twice.

## Ways to implement and run tests

Lets have a slightly more concrete example. We'll look for solutions to the quadratic equation
$$ ax^2 + bx + c =0,$$
where $x$ is a real number. The formula for the solutions is
$$ x=\frac{-b \pm \sqrt{b^2-4ac}}{2a},$$
(we can prove this by writing
$$ ax^2 + bx + c = a\left(x+\frac{b}{2a}\right)^2 -\frac{b^2}{4a}+c,$$
and rearranging).
Now to write some code in a module file

In [ ]:
import numpy as np  # for the sqrt function


def solve_quadratic(a, b, c):
    """Solve a quadratic equation ax^2+bx+c=0 in the reals"""
    if 0 < b**2 - 4.0 * a * c:
        # two solutions
        return (
            (-b - np.sqrt(b**2 - 4.0 * a * c)) / (2.0 * a),
            (-b + np.sqrt(b**2 - 4.0 * a * c)) / (2.0 * a),
        )
    elif 0 == b**2 - 4.0 * a * c:
        # one solution
        return -b / (2.0 * a)
    else:
        # no solutions
        return None

We can try some ad hoc tests, like we might run like coding the function:

In [ ]:
# solve x^2-1=0
solve_quadratic(1, 0, -1) == (-1.0, 1.0)

In [ ]:
solve_quadratic(1, 0, 0) == (0.0)

In [ ]:
solve_quadratic(1, 0, 1) is None

On the other hand, humans are lazy and forgetful, so we want to make running tests when the code changes as easy as possible. We could just roll the tests into a single function, and use the `assert` statement, so that it throws an exception if anything goes wrong.

In [ ]:
def test_solve_quadratic():
    assert solve_quadratic(1, 0, -1) == (-1.0, 1.0)
    assert solve_quadratic(1, 0, 0) == (0.0)
    assert solve_quadratic(1, 0, 1) is None

    return "Tests pass"

Remember how the `assert` statement works

In [ ]:
# does nothing
assert True

In [ ]:
# raises an exception
assert False

In [ ]:
# can set a message
assert 1 == 0, "My maths is broken!"

Avoid brackets, you might accidentally make a tuple

In [ ]:
try:
    assert 1 == 2  # ok
except AssertionError:
    print("Fix your maths!")

In [ ]:
try:
    assert (1 == 2, "My maths is wrong!")
except AssertionError:
    print("Fix your maths")

In [ ]:
# Lects actually run our tests
test_solve_quadratic()

We can do this better. There are a number of ways of automating the testing process, so that it "just happpens" without requiring the programmer to do so many things.

### The `doctest` module

The module [doctest](https://docs.python.org/2/library/doctest.html), from the standard Python library, provides a simple way to include code which is both a test and documentation of an example of the use of your code.

To write a test, one simply copies the input and output that one would see _in the vanilla python interpreter_ pretty much identically into a docstring, whether for a function or module. You've seen some of these in the assessments.

_tests\test\_docstring.py:_
```python
import doctest

def mean(x):
    """Mean of a list of numbers.

    >>> mean([1, 5, 9])
    5.0

    """
    return sum(x)/len(x)

if __name__ == "__main__":
    import doctest
    doctest.testmod()

```

In this case we can run the test by calling the module as a script:
```bash
cd tests
python -m test_docstring
```
or
```bash
python tests/docstring_test.py
```

If the test suceeds it silently returns a successful exit code. If the test fails (e.g. we replace the out put of 5.0 in the example) then an error message is printed, looking like the following:

```bash
**********************************************************************
File "docstring_test.py", line 6, in __main__.mean
Failed example:
    mean([1, 5, 9])
Expected:
    3.0
Got:
    5.0
**********************************************************************
1 items had failures:
   1 of   1 in __main__.mean
***Test Failed*** 1 failures.
```

Doctests can also be run in Python on plain text files as

```python
import doctest
doctest.testfile("example.txt")
```
or from the command line as

```bash
python -m doctest example.txt
```

### The `unittest` module

It's still useful to automate things a bit more. Python provides an inbuilt `unittest` module, which (with some work) can be used to build a test framework.

This introduces the basic concept of the three stages of a test:
1. Set up. We create anything which must already exist for a test to make sense, especially if it's expensive
2. Running. The test(s) themselves operates
3. Tear down. We clean up anything which won't get dealt with automatically

Let's say we have a module file like

_foo.py_
```python
import numpy as np  # for the sqrt function

def solve_quadratic(a, b, c):
    """Solve a quadratic equation ax^2+bx+c=0 in the reals"""
    if 0 < b**2-4.0*a*c:
        # two solutions
        return ((-b-np.sqrt(b**2-4.0*a*c))/(2.0*a),
                (-b+np.sqrt(b**2-4.0*a*c))/(2.0*a))
    elif 0 == b**2-4.0*a*c:
        # one solution
        return -b/(2.0*a)
    else:
        # no solutions
        return None

```

Suppose the file `test_unittest_example.py` contains the following code

```python
import unittest
import foo

def expensive_function():
    """Pretend this uses a lot of resources."""
    return (1, 0, -1)

def make_safe(val):
    """This might clean up memory or free up resources"""
    pass
```

```python
class TestSolveQuadratic(unittest.TestCase):

    def setUp(self):
        """Runs first."""
        # Can create values stored in the class instance self

        self.val = expensive_function()

    def test_solve_quadratic_part_one(self):

        self.assertEqual(foo.solve_quadratic(*self.val), (-1.0, 1.0))
        self.assertEqual(foo.solve_quadratic(1, 0, 0), 0.0)

    def test_solve_quadratic_part_two(self):
        self.assertEqual(foo.solve_quadratic(1, 0, 1), None)

    def tearDown(self):
        """Called at the end if we need to clean up."""
        make_safe(self.val)

# command to run the tests
unittest.main()
```

Running the test using the syntax

```bash
python tests/test_unittest_example.py
```

gives the output

```bash
..
----------------------------------------------------------------------
Ran 2 test in 0.000s

OK
```

### Pytest

The `pytest` package (not included in a default Python installation, but in the `npp2026` environment) simplifies the actions of writing tests even further, as well as providing a more informative interface. Pytest can be installed with
```bash
conda install pytest
```
or

```bash
pip install pytest
```

This also adds a tool of the same name, which can be run from the command line. This tool can be used to run both `doctest`s, (add the `--doctest-modules`) & unit tests based on the unit test module (just leave out the `unittest.main()`), as well as tests in its own format.

Look in the `tests/` folder for `test_pytest_example.py` which contains the following code

```python
import foo

def test_solve_quadratic():
    assert foo.solve_quadratic(1,0,-1)==(-1.0,1.0)
    assert foo.solve_quadratic(1,0,0)==0.0

def test_solve_quadratic_part_two():
    assert foo.solve_quadratic(1,0,1) is None

```

while the `foo.py` module contains our `solve_quadratic` function.

To run this test (remember to install pytest first) run

```bash
pytest tests/test_pytest_example.py
```

which will call our two test functions (and thus our 3 asserts).

In [ ]:
!pytest --help

Note `pytest` and `python -m pytest` do esssntially the same job.

Only major difference is whether current directory is checked for code.

#### Test discovery

By [default](https://docs.pytest.org/en/stable/explanation/goodpractices.html#test-discovery) (can change through configuration) `pytest` is set-up to scan directories sequentially looking for tests:

- Any Python file starting `test_` or ending `_test` is considered.
- Any `function starting `test` are run.
- Any methods starting `test` in classes starting `Test` are run.

These rules are sometimes hard to remember/get right. If in doubt, be explicit.

#### Running individual tests

To run just one or two tests we can call them by name:

```
pytest tests/test_pytest_example.py::test_solve_quadratic_part_two
```

We can also use the `-k` option to match by pattern

```
python -m pytest . -k part_one
```

Very useful if we've put our tests in well named classes.

### Code coverage.

In general, it is best practice to ensure that your tests exercise every line of your code at least once. This is especially true for interpretted languages like Python, which only checks the syntax of a file (i.e its "grammar") when it is first loaded, but doesn't check that the meaning makes sense.

That means you can write code like the following in a module file and it will load without error

__example.py:__
```python
def f(x):
    return y
```

but trying to actually run it will error out, in this case with a NameError.

```python
import example
example.f(1)
```

returns
```
NameError: name 'y' is not defined
```

There is a Python package, `pytest-cov`, which adds coverage information to `pytest`. Once installed with `conda` (or `pip`), coverage information can be collected for a package or module called `mycoolproject` by calling `pytest` as

```
python -m pytest --cov=mycoolproject .
```

This will generate output like
```bash
---------- coverage: platform darwin, python 3.6.8-final-0 -----------
Name                               Stmts   Miss  Cover
------------------------------------------------------
mycoolproject/__init__.py              4      0   100%
mycoolproject/analysis.py             33     24    27%
mycoolproject/live.py                 58     40    31%
mycoolproject/tests/test_example.py   11      1    91%
mycoolproject/validator.py            11      0   100%
------------------------------------------------------
TOTAL                                117     65    56%
```

Note while 100% code coverage is useful, it only means that the code has been run, not that every possible code logic branch has been tested. It is a useful, but neither necessary nor sufficient clue to healthy code.

### The many sorts of test

When developing software, opportunities (and often the need) to test appear at many different levels, and these families have developed their own names.

#### Unit tests

Unit tests are small and (ideally) quick tests which verify the behaviour of a single programming "unit" (i.e. a module or function). Thus they ensure that the unit satisfies the "contract" it makes in isolation.

Since the function is tested by itself, rather than on the job, inputs must be "mocked up" (i.e. hardcoded at compile time, or replaced with trivial substitutes) to generate known outputs and external dependencies may be replaced with "test stubs" which short circuit to quickly give the information needed by the unit being tested. The `unittest` module defines a useful class for this, called `unittest.Mock`. It also provides some other helpful functions.

#### Integration tests

Tests which combine multiple program units together and confirm that the interaction proceeds as expected. This might mean chaining multiple functions you write together, or using the real external dependencies rather than the fakes you generated for your unit tests. Integration tests tend exercise more code at once, but are slower to run and can be difficult to write effectively

#### Feature/Functionality tests

Tests which confirm an entire feature is working successfully as a whole, effectively interacting with your software as a user would. For numerical codes, this often involves analytic solutions and/or using simple "toy" problems which are well understood and have nice solutions. For more general software, it might even involve testing human interaction to make sure that GUIs are clear and robust.

#### Regression tests

Tests which check that fixed bugs stay fixed. I.e. that introducing a new change does not break the existing functionality. The tests may be at the level of the unit, integration or features. If bug reports have a minimal example attached, then this can provide the basis for useful examples for regression testing (this is a form of "red-green" testing similar to test driven development).

## Continuous integration

When working in a team, we want to make sure that everyone is working towards a single target.

If each developer has their own version of their code base, and works independently, then they will rapidly diverge, and may end up with mutually incompatible changes.

Fixing this is a source of much pain and suffering, but can be avoided by using a version control system and employing social rules to ensure that everyone is working towards the same goal, along with automated tools to catch problems early.

Continuous Integration (CI) is the philosophy that developers in a team integrate code changes into a single shared repository early and often, perhaps several times a day. Each 'push' to the repository can then be verified by automated testing tools, allowing any problems to be detected early.

From [Wikipedia](https://en.wikipedia.org/wiki/Continuous_integration):

> _In software engineering, continuous integration (CI) is the practice of merging all developers' working copies to a shared mainline several times a day. Grady Booch first proposed the term CI in his 1991 method, although he did not advocate integrating several times a day. Extreme programming (XP) adopted the concept of CI and did advocate integrating more than once per day – perhaps as many as tens of times per day._

### Automated Continuous integration

 With tests, we want to ensure changes to the code don't break them. For code under version control, this means we want to rerun the test suite on public merges, ensuring the code installs & runs successfully on a clean system. Continous integration (CI) and Continuous Delivery (CD) frameworks tie together version control repositories and test suites so that this happens automatically whenever a commit to production code is proposed, or even whenever any change is commited anywhere in the repository.

Goal is to ensure that the production branch always works (a single source of truth), so that pusers can trust they'll always be able to use it securely. Meanwhile, developers can feel safe in repeatedly merging changes to and from the production version and knowing whether their code works or not.

Extreme programming workflows stress the following points:

 - Merge back to main branch frequently (i.e keep feature changes small and atomic )
 - Automate the software build
 - Automate the testing framework

### GitHub Actions

For a concrete example, lets look at GitHub actions; a service that allows you to run tests on your code every time you push to GitHub. This is a great way to make sure that your code is always working, and that you don't break anything when you make changes.

_.github/workflows/tests.yml_:

```yml
name: tests

on: [push]

jobs:
  tests:
    runs-on: ubuntu-latest # Set OS to run on
    timeout-minutes: 30 # expected timeout
    strategy:
      matrix:
        python-version: ["3.14", "3.13"]

    steps:     # Set Python versions to test
      - uses: actions/checkout@v3
      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v3
        with:
          python-version: ${{ matrix.python-version }}

      - name: Install dependencies # Install code/dependencies
        run: |
          python -m pip install --upgrade pip
          pip install .

      - name: Run tests # A script to run (in this case pytest)
        run: |
          pytest -vv

```

In above example, we have a workflow called `tests` which runs on every push to the repository. It runs on the latest version of Ubuntu, and tests the code on Python 3.14 and 3.13. It then installs the code and dependencies, and runs the tests using `pytest`. If any of the tests fail, the workflow will fail, and you will be notified by email.

Other CI tools exist, both propriatary, public cloud solutions & opensouce:
- Travis CI
- Azure Pipelines
- CircleCI
- Jenkins
- GitLab CI
- Buildbot
How to set these up is beyond the scope of this course, but the principles are the same. You'll work more with GitHub actions in the group projects.

Let's go look on GitHub!

## Automatic documentation

Documentation is written for three broad groups of people:

1. Users ("I don't care how it does it, I care what it does.")
1. User-Developers ("I build what I need.")
1. Developers ("I make it, but I don't use it.")

These individuals have different needs from the documentation you provide to them. Users want to know what software can do, and how to run it for their own problems. Developers need to understand how code works, but not necessarily what it's intended for in a wider sense. User-developers need to understand both sides. Commercial software is generally written by pure developers, but a lot of scientific software (and many other small projects) is written by user-developers.

For all groups, the most important piece of documentation in Python is the docstring, since it remains with the code it applies to, and connects to the Python online help. For developers & user-developers, additional comments in the body of the source text may also be useful, however users will probably never choose to look there.

Although it is now generally acknowledged that the best place for living documentation is in the source code, near where it applies, since this gives the best probability of developers updating it when they changes, it is still useful to maintain a proper (electronic) manual or write-up0, both for ease of reference, and to give a general project overview.

Several tools have been developed to close this gap by automatically collating comments and function call signatures from the source code and converting it into a human readable document to label as a "manual". Perhaps the most famous open source, cross-language documentation tool is [Doxygen](http://www.doxygen.nl). However we'll look further at a tool called [Sphinx](), which originated in the Python community and is the tool of choice on several large Python projects including SciPy, Django and [Python itself](https://docs.python.org/3/about.html)

### Sphinx

You can install the core Sphinx documentation generation tools with the command

```bash
conda install sphinx
```

_or_

```bash
pip install sphinx
```

Sphinx works by converting [reStructured text](http://docutils.sourceforge.net/docs/user/rst/quickstart.html), whether inside docstrings or in special files into HTML pages or PDF files.

The simplest use pattern is probably to use the automatic scanning tools to collect together all the docstrings into an index of APIs.

This requires creating two files, the first containing the configuration options for the Sphinx toosl, and the second containing a skeleton reStructured text file into which to slot the generated documentation:

_`docs/conf.py`_:

```python
import sys
import os

## We're working in the ./docs directory, but need the package root in the path
## This command appends the directory one level up, in a cross-platform way.
sys.path.insert(0, os.path.abspath(os.sep.join((os.curdir, '..'))))

project = 'MyCoolProject'
extensions = ['sphinx.ext.autodoc',
              'sphinx.ext.napoleon',
              'sphinx.ext.mathjax']
source_suffix = '.rst'
master_doc = 'index'
exclude_patterns = ['_build']
autoclass_content = "both"
```

_`docs/index.rst`_:

```rst
#############
mycoolproject
#############


A heading
---------

This is just example text, perhaps with mathematics like :math:`x^2`, **bold text** and *italics*.
It might also include citations [1]_ inline references to functions like :func:`my_func` or even whole code blocks::

    def my_example(a, b):
        """Do something!"""
        return a**b+1

.. automodule:: mycoolproject
  :members:


.. rubric:: References
.. [1] http://www.sphinx-doc.org/en/master/usage/restructuredtext/basics.html#citations
```

With this setup we can build a `html` version of the documentation with the command

```bash
sphinx-build -b html docs docs/html
```

##### Warning

Microsoft Windows uses `\` as the separator symbol between levels in the directory tree. Meanwhile Linux and Mac OSX use `/`. This makes it almost impossible for human-readable notes to give paths that work on both sets of computers (it's a lot easier for Python code, just use `os.sep` from the `os` module, or look up the `os.path.join()` function. In this section I use the \*nix standard of `/` in the write-up, to match the Sphinx documentation, and Mac & WSL systems. Please remember to convert things in your head if working natively on a Windows computer.

If successful, you should be able to open `./docs/html/index.html` to see documentation automatically generated from the docstrings in your project. Sphinx also supports other output formats (for example LaTeX) with the `-b` flag. A recipe to generate a `pdf` manual on a suitably configured system is

```bash
sphinx-build -b latex docs .
pdflatex MyCoolProject.tex
pdflatex MyCoolProject.tex
```

which will generate `MyCoolProject.pdf`. We run LaTeX twice to ensure that references and citations (including the index) are set correctly.

## Summary

<h2> In this lecture we learned</h2>

<ul>
    <li>How to write a formal test</li>
    <li>The basics of Test Driven Development</li>
    <li>To use automated testing using testing frameworks and py.test</li>
    <li>A little CI with GitHub Actions</li>
    <li>Documentation via Sphinx</li>
</ul>

Next week: Computational Mathematics with Prof. Matthew Piggott

## Further Reading

- The Python documentation on [unittest](https://docs.python.org/3/library/unittest.html) and [doctest](https://docs.python.org/3/library/doctest.html).
- The Pytest [documentation](https://pytest.readthedocs.io/en/latest/).
- A Software Carpentry [lecture](https://intermediate-and-advanced-software-carpentry.readthedocs.io/en/latest/testing-python.html) on the theme of code testing in Python.
- More tutorials on [GitHub Actions]()
- A much fuller [Sphinx tutorial](https://pythonhosted.org/an_example_pypi_project/sphinx.html)

## This completes the lectures for Numerical Programming in Python

Thanks for your attention!

Assessment 2 due tomorrow 4.30pm

Link to module feedback form on Teams.

Your next lecture:
- 9am-12pm Computational Mathematics on Monday with Prof. Matthew Piggott across RSM 1.47/1.49/1.51/3.01D/E